# Day 2: answering the question

**Python for Data Analysis**, Coding for Reproducible Research (CfRR), University of Exeter

Yesterday we cleaned every month of Oxford weather since 1853 and found that summer 2026 tops the ranking. Today we find out whether that means anything: we plot it, fit a trend, bring in a second station, try some machine learning, and make sure the whole analysis can be rerun by someone else.

## By the end of today you will be able to

- explain why a notebook must run top to bottom from a fresh kernel
- build a matplotlib figure step by step with `fig, ax`, and save it
- make the same chart interactive with Plotly, and say when each is the better tool
- fit a straight-line trend and say what it can and cannot tell you
- turn repeated steps into a function and check it with `assert`
- use scikit-learn's `fit`/`predict`/`score` pattern, and test a model honestly

## How to use this notebook

As yesterday, there are two versions. In `day2.ipynb` most code cells are empty, and you type the code into them as the instructor types it on screen; the few cells that are filled in already, just run. [`day2_solutions.ipynb`](day2_solutions.ipynb), in the same folder, has every cell filled in, with its output: copy any cell you miss from there.

*Data: Met Office historic station data for Oxford and Lerwick. Contains public sector information licensed under the [Open Government Licence v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).*

## 1. Recap, and "Restart and run all"

*10:00, 15 minutes*

Yesterday's cleaning lives in `day2_checkpoint.ipynb`, in the same folder as this notebook. The first cell, filled in for you, runs it with `%run`, which hands us its variables, including the cleaned `oxford` DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

%run day2_checkpoint.ipynb
oxford.shape

The checkpoint also defined the file location and column names we will reuse later.

### The page is not the program

Here are yesterday's summer averages again, in two cells:

Now delete the `summer` cell and run the `summer_tmax` cell again. It still works. Then choose **Kernel > Restart Kernel and Run All Cells**, and it fails with `NameError: name 'summer' is not defined`.

The variable `summer` stayed in the kernel's memory after its cell was deleted, so everything kept working until the kernel restarted. A notebook remembers everything you have ever run, in the order you ran it, not the order on the page. The only reliable test is to restart and run everything from the top. Put the missing line back, above the cell that uses it.

## 2. Seeing it

*10:15, 45 minutes*

A matplotlib chart has two main objects: the **Figure** (the whole image) and the **Axes** (one plot inside it, with its own axes, labels and lines). Every chart here starts with `fig, ax = plt.subplots()` and then adds things to `ax`. Writing it this way takes a few more lines than a one-liner, but you control everything. The [CfRR Matplotlib page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Matplotlib.html) has many more chart types.

Start with every month in the record.

Unreadable: the seasonal cycle swamps everything else. Choose the chart for the question. Ours is about summers, so plot one value per summer.

(Why not the mean for each calendar year? 2026 only has nine months so far, all from the warmer part of the year, which would flatter it.)

Year-to-year noise hides any slow change. A 10-year rolling mean averages each summer with the nine before it. Add it as a second, thicker line, with a legend to tell them apart.

Because we kept hold of `ax`, we can keep adding to the same chart. Highlight 2026, then save the figure as a file you could put in a paper.

### The same thing with Plotly

*Last 15 minutes of this block*

Plotly makes interactive charts that you can hover over, zoom and pan. `plotly.express` builds a whole chart in one call. The [CfRR Plotly page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/data-analysis-plotly.html) goes further.

Hover over the points to find the three hottest summers. Drag across the 1970s to zoom in, and double-click to zoom back out.

Which tool, when?

- **Plotly:** quick to write and good for exploring; charts can be shared as web pages.
- **matplotlib:** more lines of code, but control over every detail, and it saves the static image files that journals want. (Plotly can save static images too, but needs the extra `kaleido` package and a copy of Chrome.)

## 3. Is it a trend?

*11:00, 25 minutes*

`np.polyfit(x, y, 1)` finds the straight line that best fits the points (degree 1 means a straight line). It returns the slope and the intercept.

Positive: summers have been warming. The x values are years, so the slope is in °C per year, and times ten gives °C per decade.

A single line through every summer since 1853 treats the whole record alike. Fit it again using only summers from 1970 onwards:

About five times steeper. A line through every summer since 1853 hides the fact that most of the warming is recent. The start year is a choice, and it changes the answer.

### Unusual for its time?

Summers are warming, so a record is less surprising than it used to be. A fairer question: how far is 2026 from what the recent trend would predict? This cell is filled in, because it has several new ideas at once. Read it a line at a time:

1. fit the line to 1970 to 2025, so 2026 cannot pull the line towards itself
2. use the line to predict 2026
3. measure the typical scatter of summers around the line
4. compare 2026's distance from the line with that scatter

In [ ]:
before = summer_tmax.loc[1970:2025]
trend_slope, trend_intercept = np.polyfit(before.index, before, 1)
expected_2026 = trend_slope * 2026 + trend_intercept
scatter = (before - (trend_slope * before.index + trend_intercept)).std()
summer_tmax[2026] - expected_2026, scatter

2026 is about two and a half times as far from the trend as a typical summer: unusual even allowing for the warming.

What a line can and cannot tell you:

- It **can** summarise the average rate of change over the years you chose.
- It **cannot** tell you why, or promise that the next ten years will follow it.
- 2026 is still **provisional**, and every 2026 temperature is estimated. The Met Office may revise them.

## Break

*11:25, 15 minutes*

## 4. A second station

*11:40, 30 minutes*

Is Oxford typical? Lerwick, in Shetland, is over 900 km further north (60.14°N, 1.18°W, 82 metres above sea level) and surrounded by sea. Its record starts in December 1930.

Yesterday's cleaning steps need to run again for a second file, so we turn them into a function, `load_station(path)`, that returns the cleaned DataFrame. You do not need to type it from scratch: copy the cleaning cell from `day2_checkpoint.ipynb`, add a `def` line, indent everything, call the DataFrame `df` and add `return df`.

Does the function reproduce yesterday's result exactly?

### Checking with `assert`

`assert` checks something you believe is true, and stops with an error if it is not. It is the simplest kind of test. A failing one looks like this:

Here are three things that should be true of any station we load:

1. no month appears twice
2. temperatures are within a plausible range for the UK
3. cleaning has not lost any values: the number of missing values in each column matches the raw file

`check_station` is filled in for you. The three `assert` lines are the part to read; the rest just gets the numbers to compare.

In [ ]:
def check_station(df, path):
    raw = pd.read_csv(path, sep=r"\s+", skiprows=7, names=COLUMNS, na_values="---")
    assert df.index.is_unique, "a month appears more than once"
    assert df["tmax"].between(-20, 40).all(), "tmax outside a plausible range"
    lost = df[MEASURES].isna().sum() - raw[MEASURES].isna().sum()
    assert (lost == 0).all(), f"cleaning lost values:\n{lost}"

check_station(oxford, OXFORD)

No output is good news for an `assert`: Oxford passes all three checks.

Now the second station. Its file is in the `data` folder too:

In [ ]:
# Snapshot of https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/lerwickdata.txt
LERWICK = DATA / "lerwickdata.txt"
LERWICK.exists()

Read the error from the bottom up: a sunshine value ending in `#` could not be read. The header of the raw file says what it means. At Lerwick, a `#` after a sunshine value means it came from an automatic sensor rather than the older recorder. Our function was written for Oxford, which never uses `#`.

A loud failure like this is a good thing: it stopped us before anything went wrong.

### The quick fix again

An AI assistant suggests a quick fix: swap `.astype(float)` for `pd.to_numeric(..., errors="coerce")`. Make that one change and run the checks:

The error goes away, but every sunshine value with a `#` silently becomes missing: years of Lerwick sunshine, gone. The third check catches it. Without the check, the code would have run and every sunshine result for Lerwick would have been wrong.

The proper fix follows yesterday's pattern: strip the `#` as well as the `*`, and keep the information in a `sun_auto` column, just as we kept the estimates.

### Long and wide data

To compare the stations, stack them into one table with a `station` column. This is **long** data: one row per station per month. Long data suits `groupby`.

Now the summer mean max for both stations, on one `ax`. Was 2026 a record summer in Lerwick too?

No. 2026 is not even in Lerwick's top three summers. The heat of 2026 was a southern story, which is exactly why one station is not enough to talk about "the UK".

The same numbers in **wide** form, one column per station, are easier to read side by side. `unstack` moves the `station` level of the index into the columns. You will also meet `pivot`, which does the same job starting from ordinary columns, and `melt`, which goes the other way, from wide back to long.

## 5. Machine learning with scikit-learn

*12:10, 40 minutes*

scikit-learn gives every model the same three methods: `fit` (learn from data), `predict` (make predictions) and `score` (measure how good they are). Learn the pattern once and every model works the same way. The [CfRR Scikit-Learn page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_ScikitLearn.html) covers more models.

### The same line, two ways

A straight-line trend is the simplest model there is. scikit-learn expects the inputs as a two-dimensional table (one row per example, one column per input), hence the `reshape`.

The same slope as `np.polyfit`. scikit-learn is the same idea with more ceremony, and the ceremony pays off once models get more complicated.

### Testing a model honestly

A model should be judged on data it has not seen. The usual approach is to hold back some data as a **test set**. `train_test_split` holds back a random quarter of the summers:

This runs, and the error looks respectable, but it is a bug that gives no error. The test summers were picked at random, so each one is surrounded by training summers on both sides. The model only has to fill in gaps, not predict the future, and the score flatters it.

With data that run through time, train on the past and test on the future. Here, train on summers before 2000 and test on 2000 onwards:

The honest error is larger. That is normal, and it is the number to report.

### Oxford or Lerwick?

Our main model is a **classifier**: given one month's weather, which station is it? The answer is a category rather than a number. We use months from 1931 onwards, when both stations recorded everything, and split by year again.

Before fitting anything, it is worth knowing what a score looks like with no model at all. The test set has as many Oxford months as Lerwick ones, so guessing at random would score 50%. A one-line rule, "if `tmax` is below 12 °C, it's Lerwick", does a little better:

These are **baselines**. A model is only worth having if it beats them. Now the model: a decision tree, which learns a series of yes/no questions about the inputs.

Clearly better than both baselines, but far from perfect. Temperature alone mixes up the seasons: a Lerwick July and an Oxford April can look alike.

What else differs between a coastal climate and an inland one? The sea keeps Lerwick's nights mild and its days cool, so the gap between day and night temperatures is small. Add that gap as a new column and refit:

A big jump. One well-chosen input, based on knowing the subject, beats a more complicated model.

## 6. Wrap: so, was summer 2026 unusual?

*12:50, 10 minutes*

This cell, filled in, pulls the numbers together:

In [ ]:
previous = summer_tmax.drop(2026)
print(f"Summer 2026 mean max: {summer_tmax[2026]:.1f} °C")
print(f"Previous record: {previous.max():.1f} °C in {previous.idxmax()}")
print(f"Above the 1970 to 2025 trend by {summer_tmax[2026] - expected_2026:.1f} °C")
print(f"Typical scatter around the trend: {scatter:.1f} °C")

So what would we write, in one sentence, and which caveats belong in it? One version:

> Summer 2026 was the hottest in Oxford's record, which goes back to 1853, by more than a degree, and well above what even the recent warming trend would predict; the figures are still provisional, and the analysis should be rerun on a later snapshot once the Met Office confirms them.

### Reproducibility checklist

- **Pinned data:** a dated snapshot of the Met Office files, kept with the notebooks; [`data/README.md`](data/README.md) records where each file came from and its checksum.
- **Locked libraries:** `uv.lock` records exact versions; `requirements.txt` is exported from it for pip users.
- **Fresh kernel:** the notebook runs top to bottom after a restart.
- **Seeds:** every random step has a `random_state`.
- **Flags kept:** estimated, provisional and automatic-sensor values are marked, not dropped.
- **Checks:** `assert` statements catch silent mistakes when the code meets new data.

### Demo: uv (optional)

uv is a fast replacement for `pip` and `venv`. In a copy of the course repository:

```bash
uv sync             # create .venv with exactly the versions in uv.lock
uv run jupyter lab  # start JupyterLab inside that environment
uv add seaborn      # add a library and update the lock file
```

If you prefer pip, `pip install -r requirements.txt` gives you the same versions.

### Where next

- [Introduction to Machine Learning](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/introduction_to_machine_learning.html)
- [Software Development Best Practices](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/software_development_best_practices.html), for proper testing
- [Version Control](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/version_control.html)